# Aula 4, Busca semântica e montagem de contexto

Notebook executável que acompanha a aula [04-busca-semantica-contexto.md](../../lessons/modulo-09-rag/04-busca-semantica-contexto.md).

## O que vamos fazer aqui

Montar o contexto com cuidado é o que transforma a busca em uma resposta confiável. Vamos
selecionar trechos acima de um limiar, numerá-los como fontes, instruir o modelo a citar e a
admitir quando não sabe. Tudo determinístico, em Python puro.

## Montagem de contexto e tratamento da ausência

Selecionamos os trechos relevantes (acima de um limiar) e montamos um prompt que pede citação das
fontes. Se nada for relevante, o prompt orienta o modelo a admitir.

In [ ]:
def montar_contexto(trechos_pontuados, limiar=0.05):
    """Seleciona trechos acima do limiar e os numera como fontes."""
    relevantes = [(s, t) for s, t in trechos_pontuados if s >= limiar]
    if not relevantes:
        return None
    return "\n".join(f"[{i+1}] {t}" for i, (s, t) in enumerate(relevantes))


def montar_prompt(pergunta, contexto):
    if contexto is None:
        return (
            "Não há material sobre a pergunta a seguir. Responda exatamente: "
            "'Não encontrei isso no material disponível.'\n"
            f"Pergunta: {pergunta}"
        )
    return (
        "Responda à pergunta usando APENAS o contexto numerado abaixo. "
        "Cite as fontes entre colchetes, como [1]. Se a resposta não estiver no "
        "contexto, diga que não encontrou.\n\n"
        f"Contexto:\n{contexto}\n\n"
        f"Pergunta: {pergunta}\nResposta:"
    )


recuperados = [
    (0.39, "A derivada mede a taxa de variação de uma função."),
    (0.21, "A regra da cadeia deriva funções compostas."),
]
print("=== Caso com material relevante ===")
print(montar_prompt("O que é a derivada?", montar_contexto(recuperados)))

print("\n=== Caso sem material relevante ===")
print(montar_prompt("Qual a capital da Mongólia?", montar_contexto([(0.01, "irrelevante")])))

No primeiro caso, o prompt traz as fontes numeradas e pede citação, produzindo uma
resposta verificável. No segundo, como nada passou do limiar, o prompt orienta o modelo a admitir
que não encontrou, evitando a alucinação. Esse tratamento honesto da ausência é essencial para a
educação. Para o projeto, implemente esse fluxo completo no assistente.